In [2]:
import numpy as np
import torch.nn as nn
import torch
import gymnasium as gym

from torch.distributions import Categorical
import torch.nn.functional as F

In [3]:
# VARIABLES
OBSERVATION_DIM = 22
ACTION_DIM = 19

NUM_ENVS = 32
TRAJECTORY_WINDOW = 128

CLIP = 0.2
VF_COEF = 0.5
ENT_COEF = 0.01
EPOCHS = 10
MB_SIZE = 256

GAMMA = 0.99
LAMBDA = 0.95

NUM_EPISODES = 25

In [4]:
class ActorCritic(nn.Module):
    def __init__(self, obs_dim, act_dim, hidden_dim=64):
        super().__init__()
        self.actor = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, act_dim),
        )

        self.critic = nn.Sequential(
            nn.Linear(obs_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, 1),
        )

    def forward(self, obs):
        logits = self.actor(obs)
        value = self.critic(obs)
        return logits, value

    def get_action(self, obs, action_mask):
        logits = self.actor(obs)
        logits = logits.masked_fill(action_mask == 0, float("-inf"))
        dist = torch.distributions.Categorical(logits=logits)
        action = dist.sample()
        log_prob = dist.log_prob(action)
        return action, log_prob

In [5]:
import sys
from pathlib import Path

# Ensure parent project folder is importable in this notebook kernel
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
	sys.path.append(str(project_root))

from core.environment import Environment
from core.configs import EnvironmentConfig

torch.manual_seed(42)

cfg = EnvironmentConfig(num_agents=2, SEE_CARD_COUNTS=True)
env = Environment(cfg)

In [6]:
ppomodel = ActorCritic(obs_dim=OBSERVATION_DIM, act_dim=ACTION_DIM)
criterion = nn.MSELoss()

In [17]:
# Collect trajectories
observations = []
actions      = []
log_probs    = []
values       = []
rewards_buf  = []
dones        = []
action_masks = []

obs, _ = env.reset()

for _ in range(TRAJECTORY_WINDOW):
    state       = obs["observation"]
    action_mask = obs["action_mask"]

    with torch.no_grad():
        action, log_prob = ppomodel.get_action(state, action_mask)
        _, value         = ppomodel(state)

    observations.append(state)
    actions.append(action)
    log_probs.append(log_prob)
    values.append(value.squeeze())
    action_masks.append(action_mask)

    obs, reward, terminated, truncated, info = env.step(action.item())
    done = terminated or truncated

    rewards_buf.append(torch.tensor(reward, dtype=torch.float32))
    dones.append(torch.tensor(float(done), dtype=torch.float32))

    if done:
        obs, _ = env.reset()

# Stack into tensors for PPO update
observations = torch.stack(observations)   # (T, obs_dim)
actions      = torch.stack(actions)        # (T,)
log_probs    = torch.stack(log_probs)      # (T,)
values       = torch.stack(values)         # (T,)
rewards_buf  = torch.stack(rewards_buf)    # (T,)
dones        = torch.stack(dones)          # (T,)
action_masks = torch.stack(action_masks)  # (T, act_dim)

In [18]:
# GAE (Generalized Advantage Estimation)
advantages = torch.zeros(TRAJECTORY_WINDOW, dtype=torch.float32)
last_gae = 0.0

with torch.no_grad():
    _, last_value = ppomodel(obs["observation"])
    last_value = last_value.squeeze()

for t in reversed(range(TRAJECTORY_WINDOW)):
    next_value          = last_value if t == TRAJECTORY_WINDOW - 1 else values[t + 1]
    next_non_terminal   = 1.0 - dones[t]
    delta               = rewards_buf[t] + GAMMA * next_value * next_non_terminal - values[t]
    last_gae            = delta + GAMMA * LAMBDA * next_non_terminal * last_gae
    advantages[t]       = last_gae

returns = advantages + values  # targets for the value function

In [9]:
returns.size(), advantages.size(), values.size()

(torch.Size([128]), torch.Size([128]), torch.Size([128]))

In [14]:
log_probs.size()

torch.Size([128])

In [ ]:
n_action, n_log_prob = ppomodel.get_action(observations, action_masks)
_, value         = ppomodel(observations)
print("Log prob:", n_log_prob.size())
print("Log probs:", log_probs.size())

ratio = n_log_prob.exp() / log_probs.exp()
left = ratio * advantages
right = torch.clamp(ratio, 1.0 - CLIP, 1.0 + CLIP) * advantages
policy_loss = -torch.min(left, right).mean()

loss = criterion(values, returns)
true_loss = policy_loss + loss.item()

true_loss
true_loss.backward()

Log prob: torch.Size([128])
Log probs: torch.Size([128])


tensor(12.7496, grad_fn=<AddBackward0>)

In [11]:
obs, _ = env.reset()
state = obs["observation"]
action_mask = obs["action_mask"]
claim_sequence = obs["claim_seq"]

print("Initial observation:", state.shape)
print("State:", action_mask.shape)

with torch.no_grad():
    logits = ppomodel.actor(state)
    print("Logits:", logits)
    masked_logits = logits.masked_fill(action_mask == 0, float("-inf"))
    print("Masked logits:", masked_logits)
    probs = torch.softmax(masked_logits, dim=-1)
    print("Action probabilities:", probs)
    dist = torch.distributions.Categorical(probs)
    action = dist.sample()
    print("Sampled action:", action.item())

    next_obs, rewards, terminated, truncated, info = env.step(action)
    print("Next observation:", next_obs)
    print("Rewards:", rewards)
    print("Terminated:", terminated)
    print("Truncated:", truncated)

Initial observation: torch.Size([22])
State: torch.Size([19])
Logits: tensor([-4.7176e-01,  6.7048e-01,  2.5657e-04, -2.1493e-01,  3.4161e-01,
        -4.7869e-01,  8.9438e-03, -4.0962e-02, -5.2489e-01, -1.2228e+00,
        -3.4518e-01,  4.2424e-01, -5.9598e-01,  9.5195e-01, -5.2389e-03,
        -3.2274e-01,  1.6485e-01,  9.1809e-02, -1.0368e+00])
Masked logits: tensor([-4.7176e-01,  6.7048e-01,  2.5657e-04, -2.1493e-01,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf,        -inf,
               -inf,        -inf,        -inf,        -inf])
Action probabilities: tensor([0.1423, 0.4458, 0.2281, 0.1839, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000, 0.0000,
        0.0000])
Sampled action: 1
Next observation: {'observation': tensor([ 0.0000,  1.0000,  0.0000,  2.0000,  2.0000,  2.0000,  3.0000,  2.0000,
         1.0000,  2.0000